# Olist E-Commerce — Customer Segmentation
**Day 5 Notebook** | Segment customers using RFM + K-Means clustering.

Pipeline: compute RFM scores → scale → find optimal k → cluster → label → connect to churn model.

## 1. Imports & Load Data

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
import joblib, os, warnings
warnings.filterwarnings('ignore')

master = pd.read_csv(
    '../processed/master_orders.csv',
    parse_dates=['order_purchase_timestamp', 'order_delivered_customer_date']
)
churn_preds = pd.read_csv('../processed/churn_predictions.csv')

delivered = master[master['order_status'] == 'delivered'].copy()
print(f'Delivered orders: {len(delivered):,}')
print(f'Churn predictions: {len(churn_preds):,}')

Delivered orders: 97,007
Churn predictions: 57,154


## 2. Compute RFM Features
**RFM** is the standard customer analytics framework:
- **Recency** — how recently did the customer last order? (lower = better)
- **Frequency** — how many orders have they placed? (higher = better)
- **Monetary** — how much have they spent in total? (higher = better)

These three numbers summarise a customer's relationship with the business.

In [2]:
# reference date: day after the last order in the dataset
ref_date = delivered['order_purchase_timestamp'].max() + pd.Timedelta(days=1)
print(f'Reference date: {ref_date.date()}')

rfm = (
    delivered
    .groupby('customer_id')
    .agg(
        recency   = ('order_purchase_timestamp', lambda x: (ref_date - x.max()).days),
        frequency = ('order_id',                 'count'),
        monetary  = ('order_value',              'sum'),
    )
    .reset_index()
)

print(f'\nRFM DataFrame shape: {rfm.shape}')
print(rfm.describe().round(2).to_string())

Reference date: 2018-08-30

RFM DataFrame shape: (96478, 4)
        recency  frequency  monetary
count  96478.00   96478.00  96478.00
mean     240.12       1.01    137.65
std      152.84       0.07    209.77
min        1.00       1.00      0.85
25%      116.00       1.00     45.90
50%      221.00       1.00     86.99
75%      350.00       1.00    149.90
max      714.00       3.00  13440.00


## 3. Visualise RFM Distributions

In [3]:
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=['Recency (days)', 'Frequency (orders)', 'Monetary (R$)']
)

colors = ['#4ade80', '#60a5fa', '#c084fc']
for i, (col, color) in enumerate(zip(['recency','frequency','monetary'], colors), 1):
    # clip outliers for visualisation
    vals = rfm[col].clip(upper=rfm[col].quantile(0.98))
    fig.add_trace(
        go.Histogram(x=vals, marker_color=color, nbinsx=50, name=col),
        row=1, col=i
    )

fig.update_layout(
    height=380, showlegend=False,
    title='RFM Feature Distributions (clipped at 98th percentile)',
    plot_bgcolor='#0c0e0a', paper_bgcolor='#111410',
    font=dict(color='#e4e8e0')
)
fig.show()

## 4. Scale Features
K-Means uses Euclidean distance — features on different scales will dominate the clustering.
StandardScaler brings all three RFM features to mean=0, std=1.

In [4]:
# log-transform monetary and frequency first — both are right-skewed
rfm['log_frequency'] = np.log1p(rfm['frequency'])
rfm['log_monetary']  = np.log1p(rfm['monetary'])

features = ['recency', 'log_frequency', 'log_monetary']
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(rfm[features])

print(f'Scaled feature matrix: {X_scaled.shape}')
print(f'Means after scaling: {X_scaled.mean(axis=0).round(4)}')
print(f'Stds after scaling:  {X_scaled.std(axis=0).round(4)}')

Scaled feature matrix: (96478, 3)
Means after scaling: [-0. -0.  0.]
Stds after scaling:  [1. 1. 1.]


## 5. Find Optimal k — Elbow + Silhouette
Two methods to choose the right number of clusters:
- **Elbow method**: plot inertia vs k — look for the elbow where gain flattens
- **Silhouette score**: measures how well-separated clusters are (higher = better)

In [5]:
from sklearn.utils import resample

SAMPLE_SIZE = 2_000
if len(X_scaled) > SAMPLE_SIZE:
    X_sample = resample(X_scaled, n_samples=SAMPLE_SIZE, random_state=42)
else:
    X_sample = X_scaled

k_range    = range(2, 11)
inertias   = []
silhouettes = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_sample)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_sample, km.labels_))

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Elbow Method (Inertia)', 'Silhouette Score']
)
fig.add_trace(
    go.Scatter(x=list(k_range), y=inertias,
               mode='lines+markers', line=dict(color='#4ade80'),
               name='Inertia'),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(x=list(k_range), y=silhouettes,
               mode='lines+markers', line=dict(color='#60a5fa'),
               name='Silhouette'),
    row=1, col=2
)
fig.update_layout(
    height=380, showlegend=False,
    title='Choosing the Right Number of Clusters',
    plot_bgcolor='#0c0e0a', paper_bgcolor='#111410',
    font=dict(color='#e4e8e0')
)
fig.show()

best_k = k_range[silhouettes.index(max(silhouettes))]
print(f'Best k by silhouette score: {best_k} (score={max(silhouettes):.4f})')
print('Look at the elbow plot — choose k where inertia stops dropping sharply.')
print('Recommended: k=4 or k=5 for interpretable business segments.')

Best k by silhouette score: 2 (score=0.8676)
Look at the elbow plot — choose k where inertia stops dropping sharply.
Recommended: k=4 or k=5 for interpretable business segments.


## 6. Fit Final K-Means Model

In [6]:
# set k — adjust if elbow/silhouette suggest different
K = 5

km_final = KMeans(n_clusters=K, random_state=42, n_init=20)
rfm['cluster'] = km_final.fit_predict(X_scaled)

sil = silhouette_score(X_scaled, rfm['cluster'])
print(f'Final model: k={K}, silhouette score={sil:.4f}')
print(f'\nCluster sizes:')
print(rfm['cluster'].value_counts().sort_index().to_string())

Final model: k=5, silhouette score=0.3339

Cluster sizes:
cluster
0    20604
1    28422
2      525
3    20304
4    26623


## 7. Profile & Label Each Cluster
Compute mean RFM values per cluster to understand who each group is.
Then assign a business label based on the profile.

In [7]:
cluster_profile = (
    rfm.groupby('cluster')
    .agg(
        n_customers  = ('customer_id',  'count'),
        avg_recency  = ('recency',      'mean'),
        avg_frequency= ('frequency',    'mean'),
        avg_monetary = ('monetary',     'mean'),
    )
    .round(1)
    .reset_index()
)

print('Cluster profiles:')
print(cluster_profile.to_string(index=False))
print()
print('Labelling guide:')
print('  Low recency + high frequency + high monetary  = Champions')
print('  Low recency + medium frequency                = Loyal Customers')
print('  High recency + any frequency                  = At Risk / Lost')
print('  Low recency + frequency=1 + low monetary      = New / One-Time Buyers')
print('  Medium recency + low frequency                = Potential Loyalists')

Cluster profiles:
 cluster  n_customers  avg_recency  avg_frequency  avg_monetary
       0        20604        389.1            1.0          46.3
       1        28422        129.7            1.0         220.2
       2          525        300.0            2.0         221.8
       3        20304        388.0            1.0         234.1
       4        26623        128.7            1.0          45.0

Labelling guide:
  Low recency + high frequency + high monetary  = Champions
  Low recency + medium frequency                = Loyal Customers
  High recency + any frequency                  = At Risk / Lost
  Low recency + frequency=1 + low monetary      = New / One-Time Buyers
  Medium recency + low frequency                = Potential Loyalists


In [8]:
# ASSIGN LABELS based on your cluster profiles above
# --- UPDATE THIS MAPPING after reading the profile table ---
# The mapping below is a placeholder — adjust cluster numbers to match your output

label_map = {
    0: 'Champions',
    1: 'Loyal Customers',
    2: 'At Risk',
    3: 'One-Time Buyers',
    4: 'New Customers',
}

rfm['segment'] = rfm['cluster'].map(label_map)

print('Segment distribution:')
print(rfm['segment'].value_counts().to_string())

Segment distribution:
segment
Loyal Customers    28422
New Customers      26623
Champions          20604
One-Time Buyers    20304
At Risk              525


## 8. Visualise Clusters

In [9]:
# 3D scatter — R, F, M
sample = rfm.sample(min(5000, len(rfm)), random_state=42)

fig = px.scatter_3d(
    sample,
    x='recency', y='frequency', z='monetary',
    color='segment',
    title='Customer Segments — RFM Space (sampled 5k)',
    labels={'recency':'Recency (days)', 'frequency':'Frequency', 'monetary':'Monetary (R$)'},
    opacity=0.6,
    height=550
)
fig.update_layout(
    paper_bgcolor='#111410', font=dict(color='#e4e8e0')
)
fig.show()

In [10]:
# segment revenue breakdown
seg_revenue = (
    rfm.groupby('segment')
    .agg(
        n_customers = ('customer_id', 'count'),
        total_spend = ('monetary',    'sum'),
        avg_spend   = ('monetary',    'mean'),
    )
    .reset_index()
)
seg_revenue['revenue_share_%'] = (
    seg_revenue['total_spend'] / seg_revenue['total_spend'].sum() * 100
).round(1)

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Customers per Segment', 'Revenue Share per Segment']
)
colors = ['#4ade80','#60a5fa','#fb923c','#c084fc','#fbbf24']
fig.add_trace(
    go.Bar(x=seg_revenue['segment'], y=seg_revenue['n_customers'],
           marker_color=colors, name='Customers'),
    row=1, col=1
)
fig.add_trace(
    go.Bar(x=seg_revenue['segment'], y=seg_revenue['revenue_share_%'],
           marker_color=colors, name='Revenue %'),
    row=1, col=2
)
fig.update_layout(
    height=400, showlegend=False,
    title='Segment Size vs Revenue Contribution',
    plot_bgcolor='#0c0e0a', paper_bgcolor='#111410',
    font=dict(color='#e4e8e0')
)
fig.show()

print(seg_revenue.to_string(index=False))

        segment  n_customers  total_spend  avg_spend  revenue_share_%
        At Risk          525    116421.25 221.754762              0.9
      Champions        20604    953791.51  46.291570              7.2
Loyal Customers        28422   6258495.08 220.198968             47.1
  New Customers        26623   1197295.55  44.972225              9.0
One-Time Buyers        20304   4753833.20 234.132841             35.8


## 9. Connect to Churn Model
Add segment labels to churn predictions — this is the cross-model connection
that shows you think end-to-end, not just notebook-to-notebook.

In [11]:
# merge segment into churn predictions
churn_seg = churn_preds.merge(
    rfm[['customer_id', 'segment', 'recency', 'frequency', 'monetary']],
    on='customer_id', how='left'
)

# churn probability by segment
seg_churn = (
    churn_seg.groupby('segment')
    .agg(
        n_customers      = ('customer_id',        'count'),
        actual_churn_rate= ('churned',            'mean'),
        avg_churn_prob   = ('churn_probability',  'mean'),
    )
    .reset_index()
    .round(3)
)

print('Churn rate by segment:')
print(seg_churn.sort_values('avg_churn_prob', ascending=False).to_string(index=False))

Churn rate by segment:
        segment  n_customers  actual_churn_rate  avg_churn_prob
Loyal Customers         8088                1.0           0.957
  New Customers         7719                1.0           0.946
One-Time Buyers        20304                1.0           0.939
      Champions        20604                1.0           0.917
        At Risk          439                0.0           0.787


In [12]:
fig = px.bar(
    seg_churn.sort_values('avg_churn_prob', ascending=False),
    x='segment', y='avg_churn_prob',
    color='avg_churn_prob',
    color_continuous_scale='RdYlGn_r',
    labels={'avg_churn_prob': 'Avg Churn Probability', 'segment': 'Segment'},
    title='Average Churn Probability by Customer Segment'
)
fig.update_layout(
    height=400,
    plot_bgcolor='#0c0e0a', paper_bgcolor='#111410',
    font=dict(color='#e4e8e0')
)
fig.show()

**Finding:** *(fill in after running)*
e.g. Champions have avg churn probability of X% vs One-Time Buyers at Y%.
At-Risk segment has the highest actual churn rate despite medium recency.

## 10. PCA Visualisation (2D)

In [13]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

rfm['pca1'] = X_pca[:, 0]
rfm['pca2'] = X_pca[:, 1]

sample_pca = rfm.sample(min(5000, len(rfm)), random_state=42)

fig = px.scatter(
    sample_pca, x='pca1', y='pca2',
    color='segment',
    title='Customer Segments — PCA 2D Projection (sampled 5k)',
    labels={'pca1': f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)',
            'pca2': f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)'},
    opacity=0.5,
    height=480
)
fig.update_layout(
    plot_bgcolor='#0c0e0a', paper_bgcolor='#111410',
    font=dict(color='#e4e8e0')
)
fig.show()

print(f'PCA explained variance: PC1={pca.explained_variance_ratio_[0]*100:.1f}%, '
      f'PC2={pca.explained_variance_ratio_[1]*100:.1f}%')

PCA explained variance: PC1=35.0%, PC2=33.5%


## 11. Save Segmentation Results

In [14]:
os.makedirs('../processed', exist_ok=True)
os.makedirs('../models', exist_ok=True)

# save full RFM + segments
rfm.to_csv('../processed/customer_segments.csv', index=False)
print(f'Saved: ../processed/customer_segments.csv ({len(rfm):,} rows)')

# save merged churn + segment file
churn_seg.to_csv('../processed/churn_with_segments.csv', index=False)
print(f'Saved: ../processed/churn_with_segments.csv ({len(churn_seg):,} rows)')

# save model + scaler
joblib.dump(km_final, '../models/kmeans_segmentation.pkl')
joblib.dump(scaler,   '../models/rfm_scaler.pkl')
print('Saved: ../models/kmeans_segmentation.pkl')
print('Saved: ../models/rfm_scaler.pkl')

Saved: ../processed/customer_segments.csv (96,478 rows)
Saved: ../processed/churn_with_segments.csv (57,154 rows)
Saved: ../models/kmeans_segmentation.pkl
Saved: ../models/rfm_scaler.pkl


In [15]:
# ── AUTO SUMMARY — runs from live variables ──────────────────────────
import pandas as pd

print('=' * 60)
print('CUSTOMER SEGMENTATION SUMMARY')
print('=' * 60)

# silhouette
print(f'\nModel: K-Means  k={K}  silhouette={sil:.4f}')
print(f'PCA explained variance: PC1={pca.explained_variance_ratio_[0]*100:.1f}%  '
      f'PC2={pca.explained_variance_ratio_[1]*100:.1f}%  '
      f'combined={sum(pca.explained_variance_ratio_)*100:.1f}%')

# cluster profile
print(f'\nCluster Profiles:')
profile = (
    rfm.groupby('segment')
    .agg(
        customers  = ('customer_id', 'count'),
        avg_recency= ('recency',     'mean'),
        avg_freq   = ('frequency',   'mean'),
        avg_spend  = ('monetary',    'mean'),
        total_spend= ('monetary',    'sum'),
    )
    .round(1)
    .reset_index()
)
profile['revenue_%'] = (profile['total_spend'] / profile['total_spend'].sum() * 100).round(1)
print(profile[['segment','customers','avg_recency','avg_freq','avg_spend','revenue_%']].to_string(index=False))

# churn by segment
if not seg_churn.empty:
    print(f'\nChurn Probability by Segment:')
    print(seg_churn.sort_values('avg_churn_prob', ascending=False).to_string(index=False))
    highest = seg_churn.loc[seg_churn['avg_churn_prob'].idxmax(), 'segment']
    lowest  = seg_churn.loc[seg_churn['avg_churn_prob'].idxmin(), 'segment']
    gap     = seg_churn['avg_churn_prob'].max() - seg_churn['avg_churn_prob'].min()
    print(f'\n  Highest risk: {highest}')
    print(f'  Lowest risk:  {lowest}')
    print(f'  Gap:          {gap*100:.1f}%')

print('\nOutputs saved:')
print('  ../processed/customer_segments.csv')
print('  ../processed/churn_with_segments.csv')
print('  ../models/kmeans_segmentation.pkl')
print('  ../models/rfm_scaler.pkl')

CUSTOMER SEGMENTATION SUMMARY

Model: K-Means  k=5  silhouette=0.3339
PCA explained variance: PC1=35.0%  PC2=33.5%  combined=68.5%

Cluster Profiles:
        segment  customers  avg_recency  avg_freq  avg_spend  revenue_%
        At Risk        525        300.0       2.0      221.8        0.9
      Champions      20604        389.1       1.0       46.3        7.2
Loyal Customers      28422        129.7       1.0      220.2       47.1
  New Customers      26623        128.7       1.0       45.0        9.0
One-Time Buyers      20304        388.0       1.0      234.1       35.8

Churn Probability by Segment:
        segment  n_customers  actual_churn_rate  avg_churn_prob
Loyal Customers         8088                1.0           0.957
  New Customers         7719                1.0           0.946
One-Time Buyers        20304                1.0           0.939
      Champions        20604                1.0           0.917
        At Risk          439                0.0           0.787

  